In [1]:
import gridlabd

In [2]:
dir(gridlabd)

['ConfigurationError',
 'GLDCheckPointMode',
 'GLDErrorCode',
 'GridLABDError',
 'GridLabD',
 'Path',
 'Simulation',
 'SimulationError',
 '__all__',
 '__builtins__',
 '__cached__',
 '__doc__',
 '__file__',
 '__loader__',
 '__name__',
 '__package__',
 '__path__',
 '__spec__',
 '__version__',
 '_lib_dir',
 '_package_dir',
 '_share_dir',
 'build_lib_dir',
 'bundle_utils',
 'get_gridlabd_info',
 'gldcore_dir',
 'glpath',
 'glpath_components',
 'gridlabd_core',
 'hello',
 'info',
 'load_model',
 'os',
 'path_sep',
 'repo_root',
 'root',
 'setup_bundled_environment',
 'simulation',
 'version']

In [3]:
gld = gridlabd.GridLabD()

In [4]:
from pathlib import Path
model_dir = Path("/mnt/c/Projects/Gridlab-d/gridlab-d/python_bindings/tests/test_HVAC_balance.glm").parent
gld.set_working_directory(str(model_dir))


Working directory set to: /mnt/c/Projects/Gridlab-d/gridlab-d/python_bindings/tests


GLDErrorCode.SUCCESS

In [5]:
gld.set_config_file("/mnt/c/Projects/Gridlab-d/gridlab-d/python_bindings/tests/gridlabd.conf")

Setting config file: /mnt/c/Projects/Gridlab-d/gridlab-d/python_bindings/tests/gridlabd.conf


GLDErrorCode.SUCCESS

In [6]:
gld.load_glm(["gridlabd", "./test_HVAC_balance.glm", "--verbose"])

GLDErrorCode.SUCCESS

In [7]:
gld.set_time_step(86400)  # Set time step to 1 day in seconds

Setting API timestep to: 86400 seconds (global_minimum_timestep unchanged at 1)


GLDErrorCode.SUCCESS

In [8]:
gld.step()

Stepping simulation forward
Simulation not initialized, attempting to initialize...

   ... initializing objects...
WARNING  [INIT] : Daylight saving time (DST) is not handled correctly when using TMY2 datasets; please use TMY3 for DST-corrected weather data.
   ... creating index 0
   ... creating index 1
   ... creating index 2
   ... shuffled 2 lists in index 0
   ... shuffled 1 lists in index 1
   ... shuffled 2 lists in index 2
Simulation initialized successfully
Stepping from time 989064000.00 to target 989150400.00 (step size: 86400 seconds)
   ... Assert passed on triplex_meter:1
   ... Assert passed on house:2
Completed step: advanced from 989064000.00 to 989150400.00


(GLDErrorCode.SUCCESS, 989150400.0)

In [ ]:
import json
# Get checkpoint as JSON string
checkpoint_json = gld.get_checkpoint_json()
checkpoint_data = json.loads(checkpoint_json)

print("Checkpoint keys:", list(checkpoint_data.keys())[:10])  # First 10 keys
print(f"Total objects: {len(checkpoint_data)}")

In [ ]:
list(checkpoint_data.keys())

In [ ]:
checkpoint_data['__preamble']

In [ ]:
checkpoint_data['clock']

In [ ]:
list(checkpoint_data['objects'])

In [ ]:
# Get current simulation time
status, current_time = gld.get_time()
print(f"Status: {status}")
print(f"Current simulation time: {current_time}")

In [ ]:
print(checkpoint_data['objects']['house'])

In [ ]:

gld.step()


In [ ]:
checkpoint_json = gld.get_checkpoint_json()
checkpoint_data = json.loads(checkpoint_json)

In [ ]:
print(checkpoint_data['objects']['house'])

In [ ]:
# Don't call exit_gld() in notebooks - it crashes the kernel
# Just let Python clean up automatically
del gld

In [1]:
# Test set_time_step behavior
import gridlabd

# Create fresh instance
gld_test = gridlabd.GridLabD()
from pathlib import Path
model_dir = Path("/mnt/c/Projects/Gridlab-d/gridlab-d/python_bindings/tests/test_HVAC_balance.glm").parent
gld_test.set_working_directory(str(model_dir))
gld_test.load_glm(["gridlabd", "./test_HVAC_balance.glm"])

print("Before set_time_step:")
status1, time1 = gld_test.get_time()
print(f"  Time: {time1}")

# Set time step to 1 day (86400 seconds)
result = gld_test.set_time_step(86400)
print(f"\nset_time_step(86400) returned: {result}")

# Step once
print("\nCalling step()...")
status2, sim_time = gld_test.step()
print(f"  Status: {status2}")
print(f"  Returned sim_time: {sim_time}")

# Check actual time
status3, time2 = gld_test.get_time()
print(f"  Actual time after step: {time2}")

# Parse times to see the difference
from datetime import datetime

def parse_gld_time(time_str):
    # Split off timezone if present
    parts = time_str.rsplit(' ', 1)
    if len(parts) == 2 and parts[1] in ['PST', 'PDT', 'EST', 'EDT', 'CST', 'CDT', 'MST', 'MDT']:
        time_str = parts[0]
    return datetime.strptime(time_str, '%Y-%m-%d %H:%M:%S')

dt1 = parse_gld_time(time1)
dt2 = parse_gld_time(time2)
diff = (dt2 - dt1).total_seconds()
print(f"\nTime difference: {diff} seconds ({diff/3600} hours, {diff/86400} days)")
print(f"Expected: 86400 seconds (1 day)")
if abs(diff - 86400) < 1:
    print("✅ SUCCESS! Stepped by exactly 1 day!")
else:
    print(f"Note: Stepping by {diff} seconds instead of 86400 (this is event-driven behavior)")


Working directory set to: /mnt/c/Projects/Gridlab-d/gridlab-d/python_bindings/tests
Before set_time_step:
  Time: 2001-05-05 05:00:00 PDT

set_time_step(86400) returned: GLDErrorCode.SUCCESS

Calling step()...
Getting current time: 2001-05-05 05:00:00 PDT
Setting API timestep to: 86400 seconds (global_minimum_timestep unchanged at 1)
Stepping simulation forward
Simulation not initialized, attempting to initialize...

WARNING  [INIT] : Daylight saving time (DST) is not handled correctly when using TMY2 datasets; please use TMY3 for DST-corrected weather data.
Simulation initialized successfully
Stepping from time 989064000.00 to target 989150400.00 (step size: 86400 seconds)
Processing 2001-05-05 22:38:10 PD  Status: GLDErrorCode.SUCCESS
  Returned sim_time: 989150400.0
  Actual time after step: 2001-05-06 05:00:00 PDT

Time difference: 86400.0 seconds (24.0 hours, 1.0 days)
Expected: 86400 seconds (1 day)
✅ SUCCESS! Stepped by exactly 1 day!
Completed step: advanced from 989064000.00 t

## Testing step_to() Function

In [ ]:
# Test step_to() function
import gridlabd
from pathlib import Path
from datetime import datetime

# Create fresh instance
gld_stepto = gridlabd.GridLabD()
model_dir = Path("/mnt/c/Projects/Gridlab-d/gridlab-d/python_bindings/tests/test_HVAC_balance.glm").parent
gld_stepto.set_working_directory(str(model_dir))
gld_stepto.load_glm(["gridlabd", "./test_HVAC_balance.glm"])

print("=== Testing step_to() Function ===\n")

# Get initial time
status1, time1 = gld_stepto.get_time()
print(f"Initial time: {time1}")

# Parse function
def parse_gld_time(time_str):
    parts = time_str.rsplit(' ', 1)
    if len(parts) == 2 and parts[1] in ['PST', 'PDT', 'EST', 'EDT', 'CST', 'CDT', 'MST', 'MDT']:
        time_str = parts[0]
    return datetime.strptime(time_str, '%Y-%m-%d %H:%M:%S')

dt1 = parse_gld_time(time1)

# Calculate target time (2 days ahead)
from datetime import timedelta
dt_target = dt1 + timedelta(days=2)
target_str = dt_target.strftime('%Y-%m-%d %H:%M:%S')

print(f"Target time: {target_str} (2 days ahead)")

# Call step_to
print(f"\nCalling step_to('{target_str}')...")
status2, final_time = gld_stepto.step_to(target_str)

print(f"\nStatus: {status2}")
print(f"Final simulation time: {final_time}")

# Verify we reached the target
status3, actual_time = gld_stepto.get_time()
print(f"Actual time from get_time(): {actual_time}")

dt_final = parse_gld_time(actual_time)
diff_from_start = (dt_final - dt1).total_seconds()
diff_from_target = (dt_final - dt_target).total_seconds()

print(f"\nTime advanced: {diff_from_start} seconds = {diff_from_start/86400:.2f} days")
print(f"Distance from target: {diff_from_target} seconds")

if abs(diff_from_target) < 1:
    print("✅ SUCCESS! Reached the target time exactly!")
elif diff_from_target >= 0:
    print(f"✅ SUCCESS! Reached or passed the target time (within acceptable range)")
else:
    print(f"❌ ERROR: Did not reach the target time")

# Cleanup
del gld_stepto